In [12]:
import os
import pandas as pd

# Path ke folder yang berisi file-file DataFrame
folder_path = '../Code/Clean_Exchange'
# Dictionary untuk menyimpan semua DataFrame dengan nama file sebagai key
df_dict = {}

# Loop melalui semua file dalam folder
for file_name in os.listdir(folder_path):
    # Pastikan file yang dibaca adalah file yang diinginkan (misalnya, CSV)
    if file_name.endswith('.csv'):
        # Buat path lengkap ke file
        file_path = os.path.join(folder_path, file_name)
        
        # Baca file dan simpan ke dictionary dengan nama file sebagai key
        df = pd.read_csv(file_path)
        df_dict[file_name] = df  # Gunakan nama file sebagai key


In [13]:
currency_dfs = {
    'MYRUSD': df_dict['MYRUSD=X.csv_clean.csv'],
    'SGDUSD': df_dict['SGDUSD=X.csv_clean.csv'],
    'THBUSD': df_dict['THBUSD=X.csv_clean.csv'],
    'USDIDR': df_dict['USDIDR=X.csv_clean.csv']
}

In [14]:
df_dict['MYRUSD=X.csv_clean.csv']

,Date,Adj Close,Close,High,Low,Open
0,2022-01-01,0.239521,0.239521,0.240038,0.239722,0.240038
1,2022-01-02,0.239521,0.239521,0.240038,0.239722,0.240038
2,2022-01-03,0.239521,0.239521,0.240038,0.239722,0.240038
3,2022-01-04,0.239808,0.239808,0.239751,0.239006,0.239406
4,2022-01-05,0.239006,0.239006,0.238949,0.238521,0.238949
...,...,...,...,...,...,...
999,2024-09-26,0.242219,0.242219,0.242872,0.241051,0.242219
1000,2024-09-27,0.241546,0.241546,0.242984,0.241692,0.241546
1001,2024-09-28,0.241956,0.241956,0.243409,0.241936,0.241956
1002,2024-09-29,0.242367,0.242367,0.243835,0.242180,0.242367


In [15]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [16]:
processed_data = {}

# Loop melalui setiap dataframe
for currency, df in currency_dfs.items():
    df = df.drop(columns=['Date'], errors='ignore')  # Drop kolom Date jika ada
    features = ['Adj Close', 'Close', 'High', 'Low', 'Open']  # Fitur yang akan digunakan
    
    # Standarisasi fitur sebelum PCA
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df[features])
    
    # PCA dengan satu komponen utama (agar tetap satu kolom per kurs)
    pca = PCA(n_components=1)
    pca_result = pca.fit_transform(scaled_data)
    
    # Simpan hasil PCA dalam dictionary
    processed_data[currency] = pca_result.flatten()

# Gabungkan semua dataset menjadi satu DataFrame
final_df = pd.DataFrame(processed_data)

In [24]:
final_df.to_csv('PCA_Exchange.csv',index=False)

## PCA FOR DATASET GLOBAL

In [18]:
import os
import pandas as pd

# Path ke folder yang berisi file-file DataFrame
folder_path = '../Code/Clean_GlobalCommo'
# Dictionary untuk menyimpan semua DataFrame dengan nama file sebagai key
df_dict = {}

# Loop melalui semua file dalam folder
for file_name in os.listdir(folder_path):
    # Pastikan file yang dibaca adalah file yang diinginkan (misalnya, CSV)
    if file_name.endswith('.csv'):
        # Buat path lengkap ke file
        file_path = os.path.join(folder_path, file_name)
        
        # Baca file dan simpan ke dictionary dengan nama file sebagai key
        df = pd.read_csv(file_path)
        df_dict[file_name] = df  # Gunakan nama file sebagai key

In [19]:
global_commo_dfs = {
    'Crude Oil WTI Futures': df_dict['Crude Oil WTI Futures Historical Data.csv_clean.csv'],
    'Natural Gas Futures': df_dict['Natural Gas Futures Historical Data.csv_clean.csv'],
    'Newcastle Coal Futures': df_dict['Newcastle Coal Futures Historical Data.csv_clean.csv'],
    'Palm Oil Futures': df_dict['Palm Oil Futures Historical Data.csv_clean.csv'],
    'US Sugar 11 Futures' : df_dict['US Sugar 11 Futures Historical Data.csv_clean.csv'],
    'US Wheat Futures' :df_dict['US Wheat Futures Historical Data.csv_clean.csv']
}

In [20]:
print(df_dict['US Sugar 11 Futures Historical Data.csv_clean.csv'].isna().sum())

Date        0
Price       0
Open        0
High        0
Low         0
Vol.        0
Change %    0
dtype: int64


In [21]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np

processed_data_commo = {}

for price, df in global_commo_dfs.items():
    df = df.drop(columns=['Date'], errors='ignore')  # Drop kolom Date jika ada
    features = ['Price', 'High', 'Low', 'Open', 'Vol.', 'Change %']

    # **1. Bersihkan angka di semua kolom numerik**
    for col in features:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)

    # **2. Tangani NaN & Inf**
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(df.median(), inplace=True)

    # **3. Standarisasi dengan StandardScaler**
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df[features])

    # **4. PCA dengan 1 komponen**
    pca = PCA(n_components=1)
    pca_result = pca.fit_transform(scaled_data)

    # **5. Simpan hasil dalam dictionary**
    processed_data_commo[price] = pca_result.flatten()

# **6. Gabungkan hasil PCA ke dalam DataFrame**
final_df_commo = pd.DataFrame(processed_data_commo)


In [23]:
final_df_commo.to_csv('PCA_global_commo.csv',index=False)
